In [4]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import matplotlib.pyplot as plt
import time
import numpy as np
from scipy.fft import fft, fftfreq

LOG.propagate = False

In [5]:
# Get ArtemisBLEController object
ble = get_ble_controller()

# Connect to the Artemis Device
ble.connect()

2026-03-04 02:11:15,449 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:42:0c:78:b8:49


2026-03-04 02:11:15,450 | INFO     |: Scanning for device with address: c0:42:0c:78:b8:49, service UUID: d427e7cc-c400-4597-b417-d564e20d6600
2026-03-04 02:11:25,558 | INFO     |: Found 1 device(s) advertising service d427e7cc-c400-4597-b417-d564e20d6600
2026-03-04 02:11:25,561 | INFO     |: Selecting device: F32F03E1-EF13-6043-59E7-D2AE8E670FC2 (name: Artemis BLE)
2026-03-04 02:11:26,386 | INFO     |: Connected to c0:42:0c:78:b8:49


In [6]:
ble.reload_config()

## ICM Sensor

In [8]:
times = []
accX = []
accY = []
accZ = []
raw_pitch = []
raw_roll = []
filt_pitch = []
filt_roll = []

def accl_notification_handler(sender, data: bytearray):
    s = ble.bytearray_to_string(data).split("|")
    if len(s) == 8:
        t = float(s[0][2:])
        x = float(s[1][2:])
        y = float(s[2][2:])
        z = float(s[3][2:])
        r_p = float(s[4][2:])
        r_r = float(s[5][2:])
        f_p = float(s[6][3:])
        f_r = float(s[7][3:])

        times.append(t)
        accX.append(x)
        accY.append(y)
        accZ.append(z)
        raw_pitch.append(r_p)
        raw_roll.append(r_r)
        filt_pitch.append(f_p)
        filt_roll.append(f_r)

        print (f"T={t:7.2f}   x={x:7.2f}   y={y:7.2f}   z={z:7.2f}   pitch={r_p:7.2f}°   roll={r_r:7.2f}°  filt_pitch={f_p:7.2f}°   filt_roll={f_r:7.2f}°")

In [ ]:
ble.start_notify(ble.uuid["RX_STRING"], accl_notification_handler)

In [ ]:
ble.stop_notify(ble.uuid["RX_STRING"])

In [ ]:
times = []
accX = []
accY = []
accZ = []
raw_pitch = []
raw_roll = []
filt_pitch = []
filt_roll = []

ble.send_command(CMD.START_COLLECT_DATA, "")
time.sleep(5)
ble.send_command(CMD.GET_ACCL_READINGS, "")

## Accelerator Pitch & Roll

In [ ]:
# Prepare arrays
t = np.array(times)
t = t - t[0]
t = t / 1000.0
r_p = np.array(raw_pitch)
r_r = np.array(raw_roll)
x = np.array(accX)
y = np.array(accY)
z = np.array(accZ)

# Create a single figure with 1x3 subplots arranged horizontally
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Pitch subplot
ax = axes[0]
ax.plot(t, p)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Pitch vs Time')
band = 10
ax.axhspan(90-band, 90+band, alpha=0.25, color='red', label='±90°')
ax.axhspan(-90-band, -90+band, alpha=0.25, color='red')
# ax.axhline(180,  color='orange', linestyle='--', linewidth=2, label='±180° bound')
# ax.axhline(-180, color='orange', linestyle='--', linewidth=2)
ax.grid(True)
ax.legend()

# Roll subplot
ax = axes[1]
ax.plot(t, r)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Roll vs Time')
# band = 10
# ax.axhspan(90-band, 90+band, alpha=0.25, color='red', label='±90°')
# ax.axhspan(-90-band, -90+band, alpha=0.25, color='red')
ax.axhline(180,  color='orange', linestyle='--', linewidth=2, label='±180° bound')
ax.axhline(-180, color='orange', linestyle='--', linewidth=2)
ax.grid(True)
ax.legend()

# Acceleration subplot
ax = axes[2]
ax.plot(t, x, label='x')
ax.plot(t, y, label='y')
ax.plot(t, z, label='z')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Acceleration')
ax.set_title('Acceleration vs Time')
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.show()

## Fourier Transform

In [ ]:
# Prepare arrays
t = np.array(times)
t = t - t[0]
t = t / 1000.0
r_p = np.array(raw_pitch)
r_r = np.array(raw_roll)
f_p = np.array(filt_pitch)
f_r = np.array(filt_roll)

# Calculate sampling rate (Hz)
dt = np.mean(np.diff(t))  # Average time step
sampling_rate = 1.0 / dt if dt > 0 else 1.0

# Number of samples
N = len(t)

# Get frequency array (only positive frequencies)
freq = fftfreq(N, dt)[:N//2]

In [ ]:
# Compute FFT for each axis
fft_raw_p = np.abs(fft(r_p))
fft_raw_r = np.abs(fft(r_r))
fft_filt_p = np.abs(fft(f_p))
fft_filt_r = np.abs(fft(f_r))

# Normalize FFT magnitudes
fft_raw_p_norm = fft_raw_p[:N//2] / N
fft_raw_r_norm = fft_raw_r[:N//2] / N
fft_filt_p_norm = fft_filt_p[:N//2] / N
fft_filt_r_norm = fft_filt_r[:N//2] / N

# Plot FFT for Pitch and Roll
fig, axes = plt.subplots(2, 2, figsize=(12,7))

# Pitch FFT
ax = axes[0, 0]
ax.plot(freq, fft_raw_p_norm, label='Raw', color='blue')
ax.plot(freq, fft_filt_p_norm, label='LPF', color='orange')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude')
ax.set_title(f'FFT of Pitch (Sampling rate: {sampling_rate:.2f} Hz)')
ax.grid(True)
ax.legend()

# Roll FFT
ax = axes[0,1]
ax.plot(freq, fft_raw_r_norm, label='Raw', color='blue')
ax.plot(freq, fft_filt_r_norm, label='LPF', color='orange')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude')
ax.set_title(f'FFT of Roll (Sampling rate: {sampling_rate:.2f} Hz)')
ax.grid(True)
ax.legend()

# Pitch Time
ax = axes[1, 0]
ax.plot(t, r_p, label='Raw', color='blue')
ax.plot(t, f_p, label='LPF', color='orange')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Pitch vs Time')
ax.grid(True)
ax.legend()

# Roll Time
ax = axes[1, 1]
ax.plot(t, r_r, label='Raw', color='blue')
ax.plot(t, f_r, label='LPF', color='orange')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Roll vs Time')
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
f_cutoff = 5.0
rc = 1.0 / (2 * np.pi * f_cutoff)
alpha = dt / (rc + dt)
print(f"Low-pass filter alpha: {alpha:.4f}")

## Gyroscope

In [ ]:
times = []
gyroR = []
gyroP = []
gyroY = []

def gyro_notification_handler(sender, data: bytearray):
    s = ble.bytearray_to_string(data).split("|")
    if len(s) == 4:
        t = float(s[0][2:])
        r = float(s[1][2:])
        p = float(s[2][2:])
        y = float(s[3][2:])
        times.append(t)
        gyroR.append(r)
        gyroP.append(p)
        gyroY.append(y)

        print (f"T={t:7.2f}   r={r:7.2f}   p={p:7.2f}   y={y:7.2f}")

In [ ]:
ble.start_notify(ble.uuid["RX_STRING"], gyro_notification_handler)

In [ ]:
times = []
gyroR = []
gyroP = []
gyroY = []

ble.send_command(CMD.START_COLLECT_DATA, "")
time.sleep(3)
ble.send_command(CMD.GET_GYRO_READINGS, "")

## Comparation

In [ ]:
ble.stop_notify(ble.uuid["RX_STRING"])

In [ ]:
ble.start_notify(ble.uuid["RX_STRING"], accl_notification_handler)

In [ ]:
times = []
accX = []
accY = []
accZ = []
raw_pitch = []
raw_roll = []
filt_pitch = []
filt_roll = []

ble.send_command(CMD.GET_ACCL_READINGS, "")

In [ ]:
# Sample frequency: 3000 microseconds (333.33 Hz)
t_f1 = t
r_p_f1 = r_p
r_r_f1 = r_r
f_p_f1 = f_p
f_r_f1 = f_r
g_p_f1 = g_p
g_r_f1 = g_r

In [ ]:
# Sample frequency: 30000 microseconds (33.33 Hz)
t_f2 = t
r_p_f2 = r_p
r_r_f2 = r_r
f_p_f2 = f_p
f_r_f2 = f_r
g_p_f2 = g_p
g_r_f2 = g_r

## Complementary Filter

In [ ]:
times = []
compR = []
compP = []


def comp_notification_handler(sender, data: bytearray):
    s = ble.bytearray_to_string(data).split("|")
    if len(s) == 3:
        t = float(s[0][2:])
        r = float(s[1][2:])
        p = float(s[2][2:])
        times.append(t)
        compR.append(r)
        compP.append(p)

        print (f"T={t:7.2f}   r={r:7.2f}   p={p:7.2f}")

In [ ]:
ble.stop_notify(ble.uuid["RX_STRING"])

In [ ]:
ble.start_notify(ble.uuid["RX_STRING"], comp_notification_handler)

In [ ]:
times = []
compR = []
compP = []

ble.send_command(CMD.GET_COMP_READINGS, "")

In [ ]:
t = np.array(times)
t = t - t[0]
t = t / 1000.0
r_p = np.array(raw_pitch)
r_r = np.array(raw_roll)
f_p = np.array(filt_pitch)
f_r = np.array(filt_roll)
g_p = np.array(gyroP)
g_r = np.array(gyroR)
c_p = np.array(compP)
c_r = np.array(compR)


SAMPLE_INTERVAL = 10000

# Plot  Pitch and Roll
fig, axes = plt.subplots(1, 2, figsize=(12,4))
title = f"Sampling rate: {1./(SAMPLE_INTERVAL*1e-6):.2f} Hz"

# Pitch Time
ax = axes[0]
ax.plot(t, r_p, label='Raw', color='blue')
ax.plot(t, f_p, label='LPF', color='orange')
ax.plot(t, g_p, label='Gyro', color='green')
ax.plot(t, c_p, label='Comp', color='purple')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Pitch vs Time')
band = 10
ax.axhspan(90-band, 90+band, alpha=0.25, color='red', label='±90°')
ax.axhspan(-90-band, -90+band, alpha=0.25, color='red')
ax.grid(True)
ax.legend()

# Roll Time
ax = axes[1]
ax.plot(t, r_r, label='Raw', color='blue')
ax.plot(t, f_r, label='LPF', color='orange')
ax.plot(t, g_r, label='Gyro', color='green')
ax.plot(t, c_r, label='Comp', color='purple')
# band = 10
# ax.axhspan(90-band, 90+band, alpha=0.25, color='red', label='±90°')
# ax.axhspan(-90-band, -90+band, alpha=0.25, color='red')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Roll vs Time')
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.show()

## Sample Data

In [ ]:
times = []

def time_notification_handler(sender, data: bytearray):
    s = ble.bytearray_to_string(data).split("|")
    if len(s) == 1:
        t = float(s[0][2:])
        times.append(t)
        print("Time(ms) =", t)

In [ ]:
ble.stop_notify(ble.uuid["RX_STRING"])

In [ ]:
ble.start_notify(ble.uuid["RX_STRING"], time_notification_handler)

In [ ]:
times = []

ble.send_command(CMD.START_COLLECT_DATA, "")
time.sleep(5)
ble.send_command(CMD.SEND_TIME_DATA, "")

In [ ]:
ble.start_notify(ble.uuid["RX_STRING"], comp_notification_handler)

In [ ]:
times = []
compR = []
compP = []

ble.send_command(CMD.START_COLLECT_DATA, "")
time.sleep(5)
ble.send_command(CMD.GET_COMP_READINGS, "")

In [ ]:
# Plot FFT
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# X-axis FFT
ax = axes[0]
ax.plot(freq, fft_x_norm, label='FFT(X)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude')
ax.set_title(f'FFT of Acceleration X (Sampling rate: {sampling_rate:.2f} Hz)')
ax.grid(True)
ax.legend()

# Y-axis FFT
ax = axes[1]
ax.plot(freq, fft_y_norm, label='FFT(Y)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude')
ax.set_title(f'FFT of Acceleration Y (Sampling rate: {sampling_rate:.2f} Hz)')
ax.grid(True)
ax.legend()

# Z-axis FFT
ax = axes[2]
ax.plot(freq, fft_z_norm, label='FFT(Z)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude')
ax.set_title(f'FFT of Acceleration Z (Sampling rate: {sampling_rate:.2f} Hz)')
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.show()

print(f"Sampling rate: {sampling_rate:.2f} Hz")
print(f"Number of samples: {N}")
print(f"Total duration: {t[-1]:.2f} s")
print(f"Peak frequency X: {freq[np.argmax(fft_x_norm[1:])+1]:.2f} Hz")
print(f"Peak frequency Y: {freq[np.argmax(fft_y_norm[1:])+1]:.2f} Hz")
print(f"Peak frequency Z: {freq[np.argmax(fft_z_norm[1:])+1]:.2f} Hz")